### Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler

In [3]:
class HeirarchicalLSTM(Dataset):
    def __init__(self, original_data, hourly_data, daily_data, weekly_data):
        self.original_data = torch.FloatTensor(original_data)
        self.hourly_data = torch.FloatTensor(hourly_data)
        self.daily_data = torch.FloatTensor(daily_data)
        self.weekly_data = torch.FloatTensor(weekly_data)

    def __len__(self):
        return len(self.original_data)
    
    def __getitem__(self, idx):
        return {
            'wsb': self.original_data[idx],
            'hourly': self.hourly_data[idx],
            'daily': self.daily_data[idx], 
            'weekly': self.weekly_data[idx],
        }

In [4]:
class HourlyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, self.hidden_size, self.num_layers, batch_first=True)

        # Attention mechanism to focus on most suspicious hours
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        # Output projection to create hourly embedding
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size = x.size(0)
        # initialize the hidden state
        h0 = torch.zeros(self.num_layers, batch_size,self.hidden_size)
        c0 = torch.zeros(self.num_layers, batch_size,self.hidden_size)

        #LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x, (h0,c0))

        # Apply attention to focus on suspicious time periods
        attended_out, attention_weights = self.attention(lstm_out,lstm_out,lstm_out)

        # Global max pooling to capture strongest coordination signals
        pooled = torch.max(attended_out, dim=1)[0]  # (batch_size, hidden_size)
        
        # Project to embedding space
        hourly_embedding = self.output_projection(pooled)
        hourly_embedding = self.dropout(hourly_embedding)
        
        return hourly_embedding, attention_weights

In [ ]:
class DailyLSTM(nn.Module):
    def __init__(self, input_size, hourly_embedding_size, hidden_size, num_layers, dropout=0.2):
        super.__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        combined_size = input_size + hourly_embedding_size
        self.lstm = nn.LSTM(combined_size, self.hidden_size, self.num_layers, batch_first=True)